# Build: Mini GPT Language Model

**Module 15** — GPT & the Scaling Revolution

> From the [Zylo Learning Platform](https://socrapy-local.vercel.app) Sequence Models course.


## Context

Let's build a mini decoder-only Transformer — a tiny GPT. This captures every key idea: token embeddings, positional embeddings, causal masked self-attention, and autoregressive generation.


## Setup

Install required packages (skip if already installed):


In [ ]:
!pip install torch -q


## Code

Run each cell in order:


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_len):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        # Causal mask — registered as buffer (not a parameter)
        self.register_buffer('mask', torch.tril(torch.ones(max_len, max_len)).unsqueeze(0).unsqueeze(0))
    
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)  # (B, T, 3*d_model)
        q, k, v = qkv.chunk(3, dim=-1)
        
        # Reshape: (B, T, d_model) -> (B, n_heads, T, d_k)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention with causal mask
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_k)
        scores = scores.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = attn @ v  # (B, n_heads, T, d_k)
        
        # Concatenate heads
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.proj(out)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, max_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))  # pre-norm variant
        x = x + self.ffn(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=4, max_len=128):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, max_len) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
    
    def forward(self, idx):
        B, T = idx.shape
        tok_emb = self.token_emb(idx)
        pos_emb = self.pos_emb(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)  # (B, T, vocab_size)
        return logits
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            logits = self(idx[:, -128:])  # crop to max_len
            logits = logits[:, -1, :] / temperature  # last position
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

# Create model
model = MiniGPT(vocab_size=256)  # character-level
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

### Step 2: Test forward pass


In [ ]:
x = torch.randint(0, 256, (2, 32))
logits = model(x)
print(f'Input shape: {x.shape}')
print(f'Output shape: {logits.shape}')  # (2, 32, 256)


---

*Generated from the [Zylo Sequence Models Course](https://socrapy-local.vercel.app). Continue learning at the platform for interactive exercises, quizzes, and AI tutoring.*
